# Bài 5 — Cây quyết định và Rừng ngẫu nhiên

**Học máy ứng dụng · Lab 05**  
Notebook thực hành theo tài liệu `Lab05_CayQuyetDinh_RungNgauNhien.pdf`.

Notebook dùng `khach_vay.csv` (phân loại trả nợ) và `gia_nha.csv` (hồi quy giá nhà). Đặt notebook cùng thư mục với hai CSV rồi chạy lần lượt từ trên xuống. Các kết quả phụ thuộc phiên bản scikit-learn, nhưng `random_state` được cố định để tái lập.

**Thư viện:** numpy, pandas, matplotlib, scikit-learn.

## 0. Chuẩn bị và đọc dữ liệu

Nhãn `tra_dung_han`: 1 là trả đúng hạn, 0 là trễ hạn. `thu_nhap` tính bằng triệu đồng mỗi tháng; `ty_le_no` là tổng nợ chia thu nhập cả năm; `co_viec_on_dinh` là biến nhị phân; `so_nguoi_phu_thuoc` từ 0 đến 4. Trong dữ liệu giá nhà, `gia` tính bằng tỷ đồng.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

# Thử thư mục hiện tại trước, sau đó thư mục data/ nếu notebook được chuyển đi.
def duong_dan_csv(ten):
    for p in (Path(ten), Path('data') / ten):
        if p.exists():
            return p
    raise FileNotFoundError(f'Không tìm thấy {ten}. Đặt CSV cùng notebook hoặc trong data/.')

khach = pd.read_csv(duong_dan_csv('khach_vay.csv'))
nha = pd.read_csv(duong_dan_csv('gia_nha.csv'))
print('Khách vay:', khach.shape)
display(khach.head())
print('Giá nhà:', nha.shape)
display(nha.head())
print(f"Tỷ lệ trả đúng hạn: {khach['tra_dung_han'].mean():.2%}")

## 1. Tự tính chỉ số Gini

Gini của một nút là $G=1-p_0^2-p_1^2$. Gini sau khi chia là trung bình có trọng số của hai nhánh. Ta thử mọi ngưỡng thu nhập giữa các giá trị liên tiếp và so với câu hỏi việc làm ổn định.

In [ ]:
nho = pd.DataFrame({
    'thu_nhap': [6, 8, 9, 11, 13, 15, 18, 20, 25, 30],
    'co_viec_on_dinh': [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
    'tra_dung_han': [0, 1, 0, 0, 1, 1, 0, 1, 1, 1],
})
y_nho = nho['tra_dung_han'].to_numpy()

def gini(nhan):
    if len(nhan) == 0:
        return 0.0
    p1 = np.mean(nhan)
    return 1 - p1**2 - (1-p1)**2

def gini_sau_chia(y, di_trai):
    trai, phai = y[di_trai], y[~di_trai]
    return len(trai)/len(y)*gini(trai) + len(phai)/len(y)*gini(phai)

g_goc = gini(y_nho)
print(f'Gini nút gốc: {g_goc:.4f}')
thu = nho['thu_nhap'].to_numpy()
muc = np.sort(np.unique(thu))
ket_qua_gini = []
for a, b in zip(muc[:-1], muc[1:]):
    nguong = (a+b)/2
    g_chia = gini_sau_chia(y_nho, thu <= nguong)
    ket_qua_gini.append((nguong, g_chia, g_goc-g_chia))
print('Ngưỡng | Gini sau chia | Gini giảm')
for nguong, g_chia, giam in ket_qua_gini:
    print(f'{nguong:6.1f} | {g_chia:13.4f} | {giam:9.4f}')
viec = nho['co_viec_on_dinh'].to_numpy()
g_viec = gini_sau_chia(y_nho, viec == 0)
print(f'
Câu hỏi việc làm: Gini sau chia={g_viec:.4f}, giảm={g_goc-g_viec:.4f}')

cay_mot_tang = DecisionTreeClassifier(max_depth=1, random_state=0)
cay_mot_tang.fit(nho[['thu_nhap', 'co_viec_on_dinh']], y_nho)
print(export_text(cay_mot_tang, feature_names=['thu_nhap', 'co_viec_on_dinh'], show_weights=True))

**Nhận xét:** Gini gốc là 0.48. Thu nhập với ngưỡng 12.0 cho mức giảm Gini lớn nhất (0.1633); câu hỏi về việc làm chỉ giảm 0.0610. Vì vậy cây chọn thu nhập làm câu hỏi đầu tiên.

## 2. Cây phân loại giới hạn năm lá

Chia dữ liệu 75/25, giữ nguyên tỷ lệ hai lớp. Cây nhỏ có thể in ra thành các luật dễ đọc; `predict_proba` là tỷ lệ lớp 1 trong lá tương ứng.

In [ ]:
X = khach.drop(columns='tra_dung_han')
y = khach['tra_dung_han']
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

cay_5la = DecisionTreeClassifier(max_leaf_nodes=5, random_state=0)
cay_5la.fit(X_tr, y_tr)
print(f'Độ sâu: {cay_5la.get_depth()}, số lá: {cay_5la.get_n_leaves()}')
print(export_text(cay_5la, feature_names=list(X.columns), show_weights=True, decimals=3))
print(f'Accuracy tập học: {accuracy_score(y_tr, cay_5la.predict(X_tr)):.4f}')
print(f'Accuracy tập kiểm tra: {accuracy_score(y_te, cay_5la.predict(X_te)):.4f}')

khach_moi = pd.DataFrame({
    'thu_nhap': [20.0, 10.0], 'ty_le_no': [0.4, 0.3],
    'co_viec_on_dinh': [1, 0], 'so_nguoi_phu_thuoc': [2, 1],
})
for i, (du_doan, xac_suat) in enumerate(zip(cay_5la.predict(khach_moi), cay_5la.predict_proba(khach_moi)[:, 1]), 1):
    print(f"Khách {i}: {'đúng hạn' if du_doan == 1 else 'trễ hạn'}, xác suất đúng hạn {xac_suat:.2f}")

fig, ax = plt.subplots(figsize=(18, 9))
plot_tree(cay_5la, feature_names=list(X.columns), class_names=['Trễ hạn', 'Đúng hạn'], filled=True, rounded=True, ax=ax)
ax.set_title('Cây quyết định giới hạn 5 lá')
plt.tight_layout()
plt.savefig('cay_nam_la.png', dpi=160, bbox_inches='tight')
plt.show()

Cây năm lá trong tài liệu có câu hỏi gốc `thu_nhap <= 15.05`. Mỗi nút `weights` cho biết số khách lớp 0 rồi lớp 1; lá dự đoán lớp chiếm đa số. Biểu đồ được lưu thành `cay_nam_la.png`.

## 3. Quá khớp và chọn độ sâu bằng kiểm định chéo

Cây tự do có thể ghi nhớ tập học. Ta chọn độ sâu bằng 5-fold cross-validation **chỉ trên tập học**, rồi đánh giá một lần trên tập kiểm tra.

In [ ]:
cay_tu_do = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
print(f'Cây tự do: sâu {cay_tu_do.get_depth()}, {cay_tu_do.get_n_leaves()} lá; '
      f'accuracy học={accuracy_score(y_tr, cay_tu_do.predict(X_tr)):.4f}; '
      f'kiểm tra={accuracy_score(y_te, cay_tu_do.predict(X_te)):.4f}')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
ket_qua_do_sau = []
for d in range(1, 16):
    mo_hinh = DecisionTreeClassifier(max_depth=d, random_state=0)
    cv = cross_val_score(mo_hinh, X_tr, y_tr, cv=skf, scoring='accuracy').mean()
    mo_hinh.fit(X_tr, y_tr)
    acc_hoc = accuracy_score(y_tr, mo_hinh.predict(X_tr))
    ket_qua_do_sau.append((d, acc_hoc, cv))
    print(f'Độ sâu {d:2d}: accuracy học={acc_hoc:.4f}, CV={cv:.4f}')

d_tot = max(ket_qua_do_sau, key=lambda r: r[2])[0]
cay_do_sau = DecisionTreeClassifier(max_depth=d_tot, random_state=0).fit(X_tr, y_tr)
print(f'
Độ sâu chọn được: {d_tot}; accuracy kiểm tra={accuracy_score(y_te, cay_do_sau.predict(X_te)):.4f}')

bang_do_sau = pd.DataFrame(ket_qua_do_sau, columns=['Độ sâu', 'Tập học', 'Kiểm định chéo'])
ax = bang_do_sau.plot(x='Độ sâu', y=['Tập học', 'Kiểm định chéo'], marker='o', figsize=(8, 4))
ax.set_ylabel('Accuracy'); ax.set_title('Accuracy theo độ sâu tối đa'); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

**Nhận xét:** Khi độ sâu tăng, accuracy tập học thường tăng liên tục trong khi điểm CV đạt đỉnh rồi giảm. Đây là dấu hiệu quá khớp; chọn độ sâu theo CV giúp cân bằng khả năng học và khái quát.

## 4. Cây không cần chuẩn hoá

So sánh cây sâu 5 trên dữ liệu gốc, dữ liệu chuẩn hoá và thu nhập đổi từ triệu sang nghìn đồng. Cây dựa trên ngưỡng từng cột nên phép đổi tỷ lệ chỉ đổi giá trị ngưỡng.

In [ ]:
def tao_cay_sau_5():
    return DecisionTreeClassifier(max_depth=5, random_state=0)

goc = tao_cay_sau_5().fit(X_tr, y_tr)
scaler = StandardScaler().fit(X_tr)
chuan = tao_cay_sau_5().fit(scaler.transform(X_tr), y_tr)
X_tr_nghin, X_te_nghin = X_tr.copy(), X_te.copy()
X_tr_nghin['thu_nhap'] *= 1000
X_te_nghin['thu_nhap'] *= 1000
nghin = tao_cay_sau_5().fit(X_tr_nghin, y_tr)

pred_goc = goc.predict(X_te)
pred_chuan = chuan.predict(scaler.transform(X_te))
pred_nghin = nghin.predict(X_te_nghin)
for ten, pred in [('Dữ liệu gốc', pred_goc), ('Chuẩn hoá', pred_chuan), ('Thu nhập tính bằng nghìn', pred_nghin)]:
    print(f'{ten:28s} accuracy={accuracy_score(y_te, pred):.4f}; khác dự đoán gốc={(pred != pred_goc).sum()}')
print(f'Ngưỡng gốc: {goc.tree_.threshold[0]:.2f}; ngưỡng thu nhập nghìn: {nghin.tree_.threshold[0]:.2f}')

## 5. Rừng ngẫu nhiên

Rừng lấy mẫu có hoàn lại cho từng cây, đồng thời giới hạn ngẫu nhiên các cột được xét ở mỗi lần chia. Các cây bỏ phiếu để giảm ảnh hưởng của những nhánh quá đặc thù.

In [ ]:
rng = np.random.default_rng(0)
n = len(X_tr)
chi_so = rng.integers(0, n, n)
so_khach_khac_nhau = len(np.unique(chi_so))
print(f'Bốc {n} lần có hoàn lại: {so_khach_khac_nhau} hồ sơ khác nhau ({so_khach_khac_nhau/n:.1%})')

rung = RandomForestClassifier(n_estimators=200, random_state=0)
rung.fit(X_tr, y_tr)
print(f'Rừng 200 cây: accuracy học={accuracy_score(y_tr, rung.predict(X_tr)):.4f}; '
      f'kiểm tra={accuracy_score(y_te, rung.predict(X_te)):.4f}')
for m in [1, 10, 50, 200, 400]:
    r = RandomForestClassifier(n_estimators=m, random_state=0).fit(X_tr, y_tr)
    print(f'{m:3d} cây: accuracy kiểm tra={accuracy_score(y_te, r.predict(X_te)):.4f}')

for ten, model in [('Cây tự do', cay_tu_do), ('Cây sâu đã chọn CV', cay_do_sau), ('Rừng 200 cây', rung)]:
    model.fit(X_tr, y_tr)
    print(f'{ten:24s} học={accuracy_score(y_tr, model.predict(X_tr)):.4f}, kiểm tra={accuracy_score(y_te, model.predict(X_te)):.4f}')

## 6. Độ quan trọng đặc trưng và phép xáo trộn

`feature_importances_` cộng mức giảm Gini theo các lần chia. Biến liên tục có nhiều ngưỡng ứng viên có thể được ưu ái; vì vậy bổ sung permutation importance trên tập kiểm tra.

In [ ]:
qt = pd.Series(rung.feature_importances_, index=X.columns).sort_values(ascending=False)
display(qt.rename('Độ quan trọng Gini').to_frame())
print(f'Tổng độ quan trọng: {qt.sum():.3f}')

# Bài minh hoạ: thêm một cột nhiễu ngẫu nhiên không liên quan nhãn.
rng = np.random.default_rng(0)
X_tr_nn = X_tr.copy()
X_te_nn = X_te.copy()
X_tr_nn['so_ngau_nhien'] = rng.random(len(X_tr_nn))
X_te_nn['so_ngau_nhien'] = rng.random(len(X_te_nn))
rung_nn = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_tr_nn, y_tr)
qt_nn = pd.Series(rung_nn.feature_importances_, index=X_tr_nn.columns).sort_values(ascending=False)
print('Gini importance khi thêm cột nhiễu:')
display(qt_nn.to_frame('Độ quan trọng Gini'))
perm = permutation_importance(rung_nn, X_te_nn, y_te, n_repeats=20, random_state=0, scoring='accuracy')
qt_perm = pd.Series(perm.importances_mean, index=X_te_nn.columns).sort_values(ascending=False)
print('Permutation importance (mức giảm accuracy trung bình):')
display(qt_perm.to_frame('Mức giảm accuracy'))
print(f"Cột nhiễu xếp hạng {qt_perm.index.get_loc('so_ngau_nhien')+1}/{len(qt_perm)} theo permutation importance.")

**Diễn giải:** độ quan trọng Gini mô tả mức mô hình dùng cột để chia, không chứng minh quan hệ nhân quả. Cột ngẫu nhiên nhiều giá trị có thể nhận điểm Gini đáng kể. Permutation importance đo mức giảm hiệu quả trên dữ liệu chưa huấn luyện khi xáo trộn cột, nên thường cho thấy cột vô ích ít ảnh hưởng.

## 7. Cây hồi quy cho dữ liệu giá nhà

Cây hồi quy chia không gian đầu vào thành vùng, và mỗi lá dự đoán trung bình giá các mẫu huấn luyện rơi vào lá. So sánh với hồi quy tuyến tính trên cùng phép chia.

In [ ]:
X_nha = nha[['dien_tich']]
y_nha = nha['gia']
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(X_nha, y_nha, test_size=0.2, random_state=42)
cay_hoi_quy = DecisionTreeRegressor(max_depth=3, random_state=0).fit(Xn_tr, yn_tr)
duong_thang = LinearRegression().fit(Xn_tr, yn_tr)
print(f'Cây sâu 3 có {cay_hoi_quy.get_n_leaves()} lá')
print(export_text(cay_hoi_quy, feature_names=['dien_tich'], decimals=1))
print(f'MAE cây: {mean_absolute_error(yn_te, cay_hoi_quy.predict(Xn_te)):.4f} tỷ đồng')
print(f'MAE tuyến tính: {mean_absolute_error(yn_te, duong_thang.predict(Xn_te)):.4f} tỷ đồng')

can_moi = pd.DataFrame({'dien_tich': [60.0, 100.0, 150.0, 250.0]})
so_sanh_du_doan = pd.DataFrame({
    'Diện tích (m²)': can_moi['dien_tich'],
    'Cây (tỷ đồng)': cay_hoi_quy.predict(can_moi),
    'Tuyến tính (tỷ đồng)': duong_thang.predict(can_moi),
})
display(so_sanh_du_doan.round(2))
luoi = pd.DataFrame({'dien_tich': np.linspace(nha.dien_tich.min(), 180, 300)})
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(nha['dien_tich'], nha['gia'], alpha=.7, label='Dữ liệu giá nhà')
ax.plot(luoi['dien_tich'], cay_hoi_quy.predict(luoi), label='Cây sâu 3')
ax.plot(luoi['dien_tich'], duong_thang.predict(luoi), label='Hồi quy tuyến tính')
ax.set(xlabel='Diện tích (m²)', ylabel='Giá (tỷ đồng)', title='Cây hồi quy và đường thẳng')
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()

**Nhận xét:** Cây tạo dự đoán bậc thang và không ngoại suy: mọi diện tích lớn hơn ngưỡng lớn nhất rơi vào cùng lá cuối, nhận cùng một giá trị. Đường hồi quy tuyến tính tiếp tục tăng theo diện tích.

# Bài tập thực hành mở rộng

Các mục dưới đây hiện thực sáu bài tập ở cuối tài liệu. Chạy các ô theo thứ tự; phần giải thích ngắn được ghi ngay sau kết quả.

## Bài 1 — Vẽ cây năm lá và cây sâu 2

Cây sâu 2 chỉ cho phép tối đa hai câu hỏi trên một đường đi. Quan sát hai lá bên trái: cây chia tiếp nếu phép chia giúp giảm impurity, dù hai lá cuối cùng có thể dự đoán cùng lớp.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 9))
plot_tree(cay_5la, feature_names=list(X.columns), class_names=['Trễ hạn', 'Đúng hạn'], filled=True, rounded=True, ax=ax)
plt.tight_layout(); plt.savefig('cay_nam_la.png', dpi=160, bbox_inches='tight'); plt.show()

cay_sau_2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_tr, y_tr)
print(export_text(cay_sau_2, feature_names=list(X.columns), show_weights=True, decimals=3))
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(cay_sau_2, feature_names=list(X.columns), class_names=['Trễ hạn', 'Đúng hạn'], filled=True, rounded=True, ax=ax)
plt.tight_layout(); plt.show()

**Trả lời:** Hai lá có thể kết luận giống nhau nhưng chứa tỷ lệ lớp và đặc điểm mẫu khác nhau. Tách nút vẫn làm giảm Gini/entropy, nên thuật toán thực hiện phép chia có lợi cho độ thuần của các nút con; cùng nhãn dự đoán không có nghĩa là hai nút có phân bố giống hệt.

## Bài 2 — Tự tìm câu hỏi Gini tốt nhất trên tập học

Thử mọi ngưỡng giữa các giá trị liên tiếp của từng cột số.

In [ ]:
def tim_cau_hoi(X, y):
    y_arr = np.asarray(y)
    g_goc = gini(y_arr)
    tot_nhat = None
    for cot in X.columns:
        muc = np.sort(X[cot].dropna().unique())
        for a, b in zip(muc[:-1], muc[1:]):
            nguong = (a+b)/2
            mask = X[cot].to_numpy() <= nguong
            g_chia = gini_sau_chia(y_arr, mask)
            ung_vien = (g_chia, cot, nguong, g_goc-g_chia)
            if tot_nhat is None or ung_vien[0] < tot_nhat[0]:
                tot_nhat = ung_vien
    return {'cot': tot_nhat[1], 'nguong': tot_nhat[2], 'gini_sau_chia': tot_nhat[0], 'gini_giam': tot_nhat[3]}

print(tim_cau_hoi(X_tr, y_tr))
print('Câu hỏi gốc cây năm lá:', X.columns[cay_5la.tree_.feature[0]], cay_5la.tree_.threshold[0])

**Kết quả kỳ vọng:** `thu_nhap` với ngưỡng 15.05 triệu đồng, trùng câu hỏi gốc của các cây phân loại trong tài liệu.

## Bài 3 — Gini so với entropy

Chọn độ sâu bằng cùng quy trình CV, rồi chấm trên cùng tập kiểm tra.

In [ ]:
ket_entropy = []
for d in range(1, 16):
    model = DecisionTreeClassifier(max_depth=d, criterion='entropy', random_state=0)
    score = cross_val_score(model, X_tr, y_tr, cv=skf, scoring='accuracy').mean()
    ket_entropy.append((d, score))
d_entropy = max(ket_entropy, key=lambda r: r[1])[0]
cay_entropy = DecisionTreeClassifier(max_depth=d_entropy, criterion='entropy', random_state=0).fit(X_tr, y_tr)
print(f'Entropy: độ sâu tốt nhất={d_entropy}, CV={max(s for _, s in ket_entropy):.4f}, kiểm tra={accuracy_score(y_te, cay_entropy.predict(X_te)):.4f}')
print(f'Gini: độ sâu tốt nhất={d_tot}, CV={max(s for _, _, s in ket_qua_do_sau):.4f}, kiểm tra={accuracy_score(y_te, cay_do_sau.predict(X_te)):.4f}')

**Nhận xét:** So sánh độ sâu và accuracy kiểm tra vừa in với Gini. Hai tiêu chí thường cho kết quả gần nhau; khác biệt nhỏ có thể đến từ cách xếp hạng các ngưỡng chia.

## Bài 4 — Cắt tỉa bằng `min_samples_leaf`

Dò số mẫu tối thiểu mỗi lá bằng CV trên tập học, rồi chấm cấu hình tốt nhất trên tập kiểm tra.

In [ ]:
gia_tri_leaf = [1, 2, 5, 10, 20, 40]
ket_leaf = []
for m in gia_tri_leaf:
    model = DecisionTreeClassifier(min_samples_leaf=m, random_state=0)
    cv = cross_val_score(model, X_tr, y_tr, cv=skf, scoring='accuracy').mean()
    ket_leaf.append((m, cv))
    print(f'min_samples_leaf={m:2d}: CV={cv:.4f}')
m_tot = max(ket_leaf, key=lambda r: r[1])[0]
cay_leaf = DecisionTreeClassifier(min_samples_leaf=m_tot, random_state=0).fit(X_tr, y_tr)
print(f'Chọn {m_tot}; accuracy kiểm tra={accuracy_score(y_te, cay_leaf.predict(X_te)):.4f}')
print(f'Cây sâu 5: accuracy kiểm tra={accuracy_score(y_te, cay_do_sau.predict(X_te)):.4f}')

## Bài 5 — So sánh độ quan trọng bằng xáo trộn

Ở mục 6 đã huấn luyện rừng có cột nhiễu và tính permutation importance trên tập kiểm tra có cột nhiễu. Hãy đọc thứ hạng `so_ngau_nhien` ở bảng kết quả và so với `feature_importances_`. Cột nhiễu có thể có Gini importance cao do có nhiều ngưỡng chia, nhưng thường làm giảm accuracy rất ít khi bị xáo trộn.

## Bài 6 — Rừng hồi quy

So sánh random forest hồi quy với cây sâu 3 và đường thẳng. Các dự đoán của cây thành phần trong rừng cũng bị chặn ở giá trị lá đã học, nên trung bình của chúng không ngoại suy tuyến tính.

In [ ]:
rung_hoi_quy = RandomForestRegressor(n_estimators=200, random_state=0).fit(Xn_tr, yn_tr)
mae_rung = mean_absolute_error(yn_te, rung_hoi_quy.predict(Xn_te))
mae_cay = mean_absolute_error(yn_te, cay_hoi_quy.predict(Xn_te))
mae_duong = mean_absolute_error(yn_te, duong_thang.predict(Xn_te))
print(f'MAE cây sâu 3: {mae_cay:.4f} tỷ đồng')
print(f'MAE rừng 200 cây: {mae_rung:.4f} tỷ đồng')
print(f'MAE tuyến tính: {mae_duong:.4f} tỷ đồng')
can_250 = pd.DataFrame({'dien_tich': [250.0]})
print(f"Dự đoán căn 250 m² — rừng: {rung_hoi_quy.predict(can_250)[0]:.2f} tỷ; tuyến tính: {duong_thang.predict(can_250)[0]:.2f} tỷ")
print(f"Giá trị lớn nhất trong tập huấn luyện: {yn_tr.max():.2f} tỷ")

**Trả lời:** Rừng hồi quy không ngoại suy: mỗi cây dự đoán giá trung bình trong lá cuối; lấy trung bình nhiều cây vẫn nằm trong phạm vi các giá trị lá/tập huấn luyện. Do đó dự đoán 250 m² bị giới hạn, trong khi tuyến tính có thể tiếp tục tăng.

## Tóm tắt

- Gini đo độ lẫn lộn và giúp tìm phép chia tốt.
- Cây nhỏ dễ giải thích; cây quá sâu dễ học thuộc tập huấn luyện.
- Chọn siêu tham số bằng kiểm định chéo trên tập học, giữ tập kiểm tra cho đánh giá cuối.
- Cây và rừng không cần chuẩn hoá theo cách KNN cần.
- Rừng thường ổn định và chính xác hơn cây đơn, còn độ quan trọng không chứng minh quan hệ nhân quả.
- Cây hồi quy và rừng không ngoại suy ra ngoài các giá trị đã quan sát.